# Case Study 01: Healthcare Claims Processing Platform
### End-to-End Medallion Architecture, Financial Data Quality, PySpark Optimization, and Databricks Workflows Orchestration

---

## Executive Summary & Business Scenario

An enterprise health insurer receives millions of medical insurance claims daily from hospitals, clinics, and specialized clinicians. The company requires reliable, auditable financial reporting and timely identification of data quality issues.

### Core Business Problem & Operational Constraints
* **Volume & Scale:** Approximately **3 TB** of incoming claims data per day.
* **SLA Bottleneck:** The existing legacy batch pipeline takes **8 hours** to complete, frequently missing executive 6:00 AM financial reporting deadlines.
* **Ledger Integrity:** Duplicate, orphaned, or missing claim records distort general ledger accounting and actuarial reserve models.
* **Reporting Requirement:** Business and finance teams require auditable, accurate **daily paid-claims reporting** by provider, member plan, and service date.
* **Idempotency & Recovery:** Failed jobs must be safely recoverable without double-counting financial disbursements or dropping legitimate claims.
* **Orchestration Standard:** Orchestrated natively via **Databricks Workflows** with automated retries, parameter propagation, repair-and-run capabilities, and automated Slack/PagerDuty incident alerting.

---

### The End-to-End Medallion Pipeline Architecture

```text
 ┌────────────────────────────────────────────────────────────────────────┐
 │                     Source Systems & S3 Landing Zone                   │
 │   Hospitals / Clinics (EDI 837 / JSON) ──> s3://claims-lake/landing/   │
 └───────────────────────────────────┬────────────────────────────────────┘
                                     │
                                     ▼
 ┌────────────────────────────────────────────────────────────────────────┐
 │               BRONZE LAYER (Databricks Auto Loader)                    │
 │   - Immutable append-only ingestion: raw payload + source file metadata│
 │   - Schema hints, bad-records routing, and ingestion batch audit IDs   │
 └───────────────────────────────────┬────────────────────────────────────┘
                                     │
                                     ▼
 ┌────────────────────────────────────────────────────────────────────────┐
 │            SILVER LAYER (PySpark Cleansing & Quarantine)               │
 │   - Data contract enforcement & schema validation                      │
 │   - Quarantine routing (missing PKs, unparseable amounts)              │
 │   - Window-based deduplication (row_number over latest updated_at)     │
 │   - Adjudication logic: Reversals, adjustments, and grain alignment    │
 │   - Transactional Delta Lake MERGE INTO (idempotent upserts)           │
 └───────────────────────────────────┬────────────────────────────────────┘
                                     │
                                     ▼
 ┌────────────────────────────────────────────────────────────────────────┐
 │               GOLD LAYER (Curated Dimensional Marts)                   │
 │   - Provider daily paid summaries (fact_claims_daily_provider)         │
 │   - Member utilization & plan analytics (fact_member_utilization)      │
 │   - Financial reconciliation ledger (source vs target paid variance)   │
 └───────────────────────────────────┬────────────────────────────────────┘
                                     │
                                     ▼
 ┌────────────────────────────────────────────────────────────────────────┐
 │            CONSUMPTION & ORCHESTRATION (Databricks Workflows)          │
 │   - Power BI Direct Lake / Databricks SQL Serverless Dashboards        │
 │   - DQ Gatekeeper & automated reconciliation validation                │
 │   - Multi-task DAG, taskValues parameter passing, Repair-and-Rerun     │
 └────────────────────────────────────────────────────────────────────────┘
```

---

### Table of Contents
1. **Business Scenario & The Claims Adjudication Lifecycle**
2. **Understand the Source Data & The Header vs. Line Grain Distinction**
3. **Design the End-to-End Medallion Architecture**
4. **Define Business Metrics Before Writing Code**
5. **Data Quality & Financial Reconciliation (The 1M Record Batch Drill)**
6. **SQL Interview Exercise & The 6 Critical Follow-Up Scenarios**
7. **PySpark Window Deduplication & Execution Internals**
8. **Incremental Loads, Delta MERGE, and Idempotency Under Failure**
9. **Performance Optimization: Reducing 8 Hours to Under 30 Minutes on 3 TB**
10. **Databricks Workflows Production Job Definition & Senior DE Interview Playbook**


In [2]:
import sys, pathlib, os, json, time, datetime, math
import pandas as pd
from pyspark.sql import functions as F, types as T, Window

# Locate repo root and register src/ directory
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "de_helpers.py").exists())
sys.path.insert(0, str(ROOT / "src"))

from de_helpers import (
    q, q1, assert_grain, tables, columns, explain, display,
    get_spark, get_dbutils, plan_string, has_plan_operator
)

spark = get_spark("case-study-healthcare-claims")
dbutils = get_dbutils()

print(f"Setup complete. Root: {ROOT}")
print("Active Tables in Practice Warehouse:")
display(tables())


Setup complete. Root: m:\databricks\databricks-wanderbricks\de-interview-prep
Active Tables in Practice Warehouse:


,table_schema,table_name,table_type
0,main,claim_services,VIEW
1,main,claims,VIEW
2,main,diagnoses,VIEW
3,main,members,VIEW
4,main,payments,VIEW
5,main,plans,VIEW
6,main,procedures,VIEW
7,main,provider_specialty_history,VIEW
8,main,providers,VIEW


---
## Section 1: Business Scenario & The Claims Adjudication Lifecycle

### The Domain Mechanics of Healthcare Claims
In health insurance, processing a medical claim is an auditable financial transaction. Understanding the lifecycle is essential for designing data models and reconciliation rules:

```mermaid
graph TD
    A[Provider submits claim EDI 837] --> B[Pre-adjudication Validation: Member coverage & provider eligibility]
    B --> C[Pricing Engine: Contracted fee schedules & fee-for-service]
    C --> D[Benefit Determination: Deductibles, copays, coinsurance applied]
    D --> E{Adjudication Outcome}
    E -->|Approved| F[Payment Scheduled: ACH 835 remittance generated]
    E -->|Denied| G[Rejection Notice: Denial reason code logged]
    E -->|Pending| H[Medical Review: Held in transit for authorization]
    F --> I[Retroactive Adjustment / Reversal: Clawbacks & audit corrections]
```

### The 5 Operational Pillars Expected of the Data Engineer:
1. **Volume Handling (3 TB Daily):** Cannot rely on full daily re-scans. Ingestion must be incremental, partitioned, and clustered.
2. **SLA Compression (8h to <30m):** Optimize execution plans, eliminate shuffle spills, broadcast dimension joins, and use Delta Liquid Clustering/Z-ordering.
3. **Ledger Integrity:** Distinguish between genuine duplicate records vs legitimate subsequent revisions (e.g. claim versioning).
4. **Reconciliation Discipline:** Compare input record counts, processed IDs, and gross/net dollar totals at every layer before releasing data.
5. **Idempotent Recovery:** Mid-job crashes must be restartable via **Databricks Workflows Repair-and-Rerun** without creating phantom disbursements.


In [ ]:
# Baseline Data Health & Volume Verification
print("--- Baseline Claims Table Summary ---")
summary_df = q("""
    SELECT 
        COUNT(*) AS total_records,
        COUNT(DISTINCT claim_id) AS distinct_claims,
        COUNT(DISTINCT provider_id) AS distinct_providers,
        COUNT(DISTINCT member_id) AS distinct_members,
        MIN(claim_date) AS min_service_date,
        MAX(claim_date) AS max_service_date,
        ROUND(SUM(billed_amount), 2) AS total_billed,
        ROUND(SUM(paid_amount), 2) AS total_paid
    FROM claims
""")
display(summary_df)

print("\n--- Claims by Status Distribution ---")
status_dist = q("""
    SELECT 
        status,
        COUNT(*) AS claim_count,
        ROUND(SUM(paid_amount), 2) AS total_paid,
        ROUND(AVG(paid_amount), 2) AS avg_paid
    FROM claims
    GROUP BY status
    ORDER BY claim_count DESC
""")
display(status_dist)


---
## Section 2: Understand the Source Data & The Header vs. Line Grain Distinction

### Source Data Schemas

| Dataset | Granularity | Key Columns | Financial Semantics |
| :--- | :--- | :--- | :--- |
| **`claims`** | **Claim Header** (1 row per claim) | `claim_id` (PK), `member_id`, `provider_id`, `service_date`, `paid_amount`, `status` | Contains final adjudicated claim-level payment and overall status. |
| **`claim_lines`** (`claim_services`) | **Claim Line** (1 row per service) | `claim_id` (FK), `line_id` / `service_id` (PK), `procedure_code`, `service_amount`, `units` | Detailed procedures (CPT/HCPCS codes) administered during the visit. |
| **`members`** | **Member** (1 row per member) | `member_id` (PK), `plan_id`, `effective_date`, `end_date` | Insurance policy enrollment and coverage timeline. |
| **`providers`** | **Provider** (1 row per provider) | `provider_id` (PK), `provider_type`, `provider_name` | Hospital, clinic, or physician directory details. |

---

### The Critical Grain Trap: Fan-Out Multi-Counting

> [!CAUTION]
> **The #1 Financial Bug in Healthcare Data Engineering:**
> A single claim header can have $N$ claim service lines ($1:N$). If an engineer naively joins `claims` to `claim_lines` and performs `SUM(claims.paid_amount)`, **the payment is multiplied by $N$**!
>
> For example: An emergency room claim has `paid_amount = $5,000` with 6 procedure lines (triage, X-ray, lab, medication, suture, facility fee).
> Naive join: $5,000 \times 6 = \$30,000$ paid amount! Your financial dashboard reports **$25,000 of phantom money**.

### Architectural Rule: Grain Isolation
* **Header Metrics** (`paid_amount`, claim counts, status counts) must be calculated **exclusively at the header grain**.
* **Line Metrics** (`service_amount`, procedure utilization, units) belong at the line grain.
* If a reporting table requires both, **pre-aggregate the lines to the header grain first** before joining, or allocate header payments down to lines using documented allowance weights.


In [ ]:
# Live Demonstration of the Fan-Out Trap Bug vs. Grain-Safe Pattern

# 1. Authoritative Header Total
header_total = q1("""
    SELECT ROUND(SUM(paid_amount), 2)
    FROM claims
    WHERE status = 'Approved'
""")

# 2. Naive Join Total (THE BUG)
naive_joined_total = q1("""
    SELECT ROUND(SUM(c.paid_amount), 2)
    FROM claims c
    INNER JOIN claim_services cs ON c.claim_id = cs.claim_id
    WHERE c.status = 'Approved'
""")

# 3. Line Count Multiplicity
line_multiplicity = q("""
    SELECT 
        c.claim_id,
        c.paid_amount AS header_paid,
        COUNT(cs.service_id) AS line_count,
        ROUND(c.paid_amount * COUNT(cs.service_id), 2) AS phantom_paid_if_summed
    FROM claims c
    INNER JOIN claim_services cs ON c.claim_id = cs.claim_id
    WHERE c.status = 'Approved'
    GROUP BY c.claim_id, c.paid_amount
    HAVING COUNT(cs.service_id) > 1
    LIMIT 5
""")

print(f"Authoritative Header Paid Amount:  ${header_total:,.2f}")
print(f"Naive Joined Total (Fan-out Bug): ${naive_joined_total:,.2f}")
print(f"Phantom Inflation Variance:       ${(naive_joined_total - header_total):,.2f} (+{((naive_joined_total/header_total - 1)*100):.1f}%)")
print("\nSample Multi-Line Claims Causing Fan-Out:")
display(line_multiplicity)

# Verify grain using assert_grain helper
print("\nGrain Assertions:")
assert_grain(q("SELECT claim_id FROM claims"), "claim_id")
assert_grain(q("SELECT claim_id, service_id FROM claim_services"), "claim_id", "service_id")


---
## Section 3: Design the End-to-End Medallion Architecture

### The 8-Step Pipeline Execution Lifecycle

1. **Ingest (Bronze):** Ingest raw landing files from S3 using **Databricks Auto Loader** (`cloudFiles`). Append raw payloads with metadata (`_ingested_at`, `_source_file`, `_batch_id`).
2. **Validate (Data Contracts):** Enforce strict typing, required fields (non-null `claim_id`), and format rules. Malformed rows route to a Quarantine table.
3. **Deduplicate (Silver Staging):** Apply window-based deduplication on `(claim_id)` ordering by `updated_at DESC, _ingested_at DESC`.
4. **Transform & Reconcile Reversals:** Net out adjustments, voided claims, and clawbacks. Ensure referential integrity against `dim_provider` and `dim_member`.
5. **Upsert into Silver (Delta Lake `MERGE INTO`):** Transactionally merge cleansed claims into Silver tables with ACID guarantees.
6. **Aggregate (Gold Marts):** Build grain-safe aggregate data marts (provider daily summaries, member utilization) optimized for BI consumption.
7. **Reconcile (Ledger Validation):** Execute automated financial reconciliation: compare source ingested paid dollars against target Gold paid dollars.
8. **Publish & Gatekeep:** Gate publication to downstream Power BI and Databricks SQL Warehouses. If variance exceeds $0.00, trigger automated alerts and hold publication.

---

### Databricks Workflows Orchestration DAG

Databricks Workflows orchestrates the entire Medallion sequence with native task dependency management:

```mermaid
graph LR
    T1[Task 1: ingest_s3_to_bronze] --> T2[Task 2: quarantine_and_silver_merge]
    T2 --> T3[Task 3: adjudicate_reversals_referential]
    T3 --> T4[Task 4: build_gold_daily_marts]
    T4 --> T5[Task 5: dq_reconciliation_gatekeeper]
    T5 -->|Variance > $0 OR Task Failure| T6[Task 6: alert_incident_webhook]
```


In [ ]:
# Simulating Medallion Bronze -> Silver Pipeline in PySpark

# 1. Ingestion Simulation (Bronze DataFrame with Audit Metadata)
raw_claims_df = spark.table("claims").withColumn(
    "_ingest_batch_id", F.lit("BATCH_2026_10_11_001")
).withColumn(
    "_ingested_at", F.current_timestamp()
).withColumn(
    "_source_file", F.lit("s3://claims-lake/landing/2026/10/11/claims_batch_001.json.gz")
)

# 2. Data Contract Rules (Silver Valid vs Quarantine Split)
# Required: non-null claim_id, non-null member_id, valid status, non-negative billed_amount
is_valid_pk = F.col("claim_id").isNotNull() & (F.length(F.trim(F.col("claim_id"))) > 0)
is_valid_status = F.col("status").isin("Approved", "Denied", "Submitted", "Pending")
is_valid_amount = F.col("billed_amount").isNotNull() & (F.col("billed_amount") >= 0)

valid_condition = is_valid_pk & is_valid_status & is_valid_amount

silver_valid_df = raw_claims_df.filter(valid_condition)
quarantine_df = raw_claims_df.filter(~valid_condition).withColumn(
    "quarantine_reason",
    F.when(~is_valid_pk, "MISSING_PRIMARY_KEY")
     .when(~is_valid_status, "INVALID_CLAIM_STATUS")
     .when(~is_valid_amount, "INVALID_BILLED_AMOUNT")
     .otherwise("UNKNOWN_VALIDATION_FAILURE")
)

print(f"Raw Ingested Records: {raw_claims_df.count():,}")
print(f"Silver Valid Records:   {silver_valid_df.count():,}")
print(f"Quarantined Records:    {quarantine_df.count():,}")


---
## Section 4: Define Business Metrics Before Writing Code

In an interview, **do not jump into SQL or PySpark immediately**. Interviewers test whether you understand the financial and actuarial nuances of the business.

| Metric | Business Definition | Formula / Implementation | Critical Semantic Considerations |
| :--- | :--- | :--- | :--- |
| **Total Claims** | Count of distinct eligible claims | `COUNT(DISTINCT claim_id)` | Exclude test claims, purged records, and voided claims. |
| **Total Paid Amount** | Net sum of insurance payments at header grain | `SUM(paid_amount)` on header | Must net out retroactive reversals and claim clawbacks. |
| **Paid Claims** | Distinct count of claims meeting paid status | `COUNT(DISTINCT CASE WHEN status = 'PAID' THEN claim_id END)` | Agree on status semantics (`PAID` vs `Approved` vs `Settled`). |
| **Average Paid per Claim** | Average reimbursement per claim | `Total Paid Amount / Eligible Claims` | **The Denominator Trap:** Do zero-paid claims (denied or 100% deductible applied) belong in the denominator? (Document policy explicitly!). |
| **Rejected Claim Rate** | Proportion of adjudicated claims denied | `Rejected Claims / (Paid Claims + Rejected Claims)` | **Pending Claim Trap:** Never include `PENDING` claims in denominator; they have not finished adjudication! |
| **Reconciliation Variance**| Source vs. Target financial gap | `Source Paid Amount - Target Ledger Paid Amount` | Must equal $0.00. Any variance triggers a production audit. |

> [!TIP]
> **Senior DE Interview Talking Point:**
> *"A technically correct `SUM()` or `AVG()` can still be a complete business failure if the grain is wrong, pending claims pollute the denominator, or reversals are double-counted as new claims."*


In [ ]:
# Accurate Business Financial Metrics Calculation in DuckDB SQL

metrics_sql = """
WITH adjudicated_population AS (
    SELECT
        claim_id,
        provider_id,
        status,
        COALESCE(paid_amount, 0.0) AS paid_amount,
        CASE WHEN status = 'Approved' THEN 1 ELSE 0 END AS is_paid,
        CASE WHEN status = 'Denied' THEN 1 ELSE 0 END AS is_rejected,
        CASE WHEN status = 'Submitted' THEN 1 ELSE 0 END AS is_pending
    FROM claims
)
SELECT
    COUNT(DISTINCT claim_id) AS total_claims_received,
    COUNT(DISTINCT CASE WHEN is_pending = 0 THEN claim_id END) AS total_adjudicated_claims,
    
    -- Paid Metrics
    COUNT(DISTINCT CASE WHEN is_paid = 1 THEN claim_id END) AS paid_claims_count,
    ROUND(SUM(CASE WHEN is_paid = 1 THEN paid_amount ELSE 0 END), 2) AS total_paid_amount,
    
    -- Average Paid per Claim (Excluding zero-paid rejected/pending)
    ROUND(
        SUM(CASE WHEN is_paid = 1 THEN paid_amount ELSE 0 END) / 
        NULLIF(COUNT(DISTINCT CASE WHEN is_paid = 1 THEN claim_id END), 0),
        2
    ) AS avg_paid_per_paid_claim,
    
    -- Average Paid across all adjudicated (including $0 denials)
    ROUND(
        SUM(CASE WHEN is_paid = 1 THEN paid_amount ELSE 0 END) / 
        NULLIF(COUNT(DISTINCT CASE WHEN is_pending = 0 THEN claim_id END), 0),
        2
    ) AS avg_paid_all_adjudicated,
    
    -- Rejected Claim Rate (Excluding pending from denominator)
    ROUND(
        SUM(is_rejected) * 100.0 / 
        NULLIF(SUM(is_paid + is_rejected), 0),
        2
    ) AS rejected_claim_rate_pct
FROM adjudicated_population;
"""

display(q(metrics_sql))


---
## Section 5: Data Quality & Financial Reconciliation (The 1M Record Batch Drill)

### The Production Scenario
Suppose today's incoming batch lands in S3 with the following characteristics:
* **1,000,000** total source records received.
* **2,000** duplicate records (re-sent claims or re-try payloads).
* **500** records with missing `claim_id` (null primary keys).
* **300** records with invalid/unparseable `paid_amount`.
* **Financial Mismatch:**
  * Source Ingested Paid Total: **$12,500,000**
  * Target Curated Paid Total: **$12,490,000**
  * **Gross Variance: $10,000**

---

### The Data Quality Action Plan & Data Contracts

| Anomaly | Root Cause | Action Taken | Architectural Destination |
| :--- | :--- | :--- | :--- |
| **Missing `claim_id` (500)** | Upstream API integration failure / unmapped JSON | Quarantine immediately. Do not generate synthetic IDs. Alert upstream partner. | `quarantine.claims_missing_pk` |
| **Duplicate Records (2,000)** | Upstream network retries or batch re-delivery | Apply deterministic deduplication (`ROW_NUMBER()` over `updated_at DESC`). | Deduplicated to Silver; rejected copies logged to audit. |
| **Invalid Amount (300)** | Format corruption (text in decimal field) | Quarantine record; do not default to $0 (would distort financial averages). | `quarantine.claims_schema_violation` |
| **Negative Paid Amount** | Legitimate reversal vs. corruption | Check claim type: If `REVERSAL` or `ADJUSTMENT`, process as offset; if original `INITIAL` claim, quarantine. | Processed or Quarantined based on transaction code. |
| **$10,000 Variance** | Source vs Target financial gap | **Halt Gold release.** Investigate reconciliation ledger exceptions. | `audit.reconciliation_exceptions` |

---

### Root Cause Localization of the $10,000 Variance

```text
Reconciliation Audit Walkthrough:
1. Source Ingested Paid Amount:                 $12,500,000.00
2. Less: Quarantined Records with Malformed Amount:   -$2,400.00 (300 bad records)
3. Less: Duplicate Claims Removed:                    -$7,600.00 (10 claims of $760 each)
4. Expected Valid Target Paid Amount:           $12,490,000.00
5. Target Silver Table Sum:                     $12,490,000.00
----------------------------------------------------------------------
Net Unexplained Variance:                                $0.00  ==> RECONCILED!
```
*Notice:* The $10,000 variance was **not lost money**—it was exactly $2,400 in quarantined malformed claims plus $7,600 in duplicate payments that were rightly dropped!


In [ ]:
# Executing the Financial Reconciliation Ledger Drill in PySpark

# Simulate the 1M batch parameters using synthetic distribution
print("=== Simulating 1,000,000 Ingested Batch Processing ===")

batch_stats = {
    "source_records_received": 1000000,
    "duplicate_records": 2000,
    "missing_pk_records": 500,
    "malformed_amount_records": 300,
    "source_paid_gross": 12500000.00,
    "quarantined_amount": 2400.00,
    "deduplicated_amount": 7600.00,
}

target_valid_records = (
    batch_stats["source_records_received"]
    - batch_stats["duplicate_records"]
    - batch_stats["missing_pk_records"]
    - batch_stats["malformed_amount_records"]
)
expected_target_paid = (
    batch_stats["source_paid_gross"]
    - batch_stats["quarantined_amount"]
    - batch_stats["deduplicated_amount"]
)

# Build Reconciliation Ledger DataFrame
recon_ledger = pd.DataFrame([
    {"Metric": "Total Records", "Source Ingested": 1000000, "Quarantined/Dropped": 2800, "Target Curated": target_valid_records, "Variance": 0},
    {"Metric": "Valid Claim IDs", "Source Ingested": 997500, "Quarantined/Dropped": 0, "Target Curated": 997500, "Variance": 0},
    {"Metric": "Paid Amount ($)", "Source Ingested": 12500000.00, "Quarantined/Dropped": 10000.00, "Target Curated": expected_target_paid, "Variance": 0.00}
])

display(recon_ledger)

# Build Reconciliation Exceptions Audit Record
reconciliation_exception_record = {
    "batch_id": "BATCH_2026_10_11_001",
    "source_total": 12500000.00,
    "target_total": expected_target_paid,
    "gross_variance": 10000.00,
    "quarantine_reasons": {
        "MISSING_CLAIM_ID": {"count": 500, "dollars": 0.00},
        "MALFORMED_AMOUNT": {"count": 300, "dollars": 2400.00},
        "DEDUPLICATED_COPIES": {"count": 2000, "dollars": 7600.00}
    },
    "unexplained_net_variance": 0.00,
    "reconciliation_status": "PASSED_WITH_AUDIT_EXCEPTIONS"
}
print("\n--- Production Reconciliation Audit Payload ---")
print(json.dumps(reconciliation_exception_record, indent=2))


---
## Section 6: SQL Interview Exercise & The 6 Follow-Up Questions

### The Primary Interview Question
> **Question:** Calculate daily paid claims and total paid amount for each healthcare provider.
> Assume `claims` has one row per claim with the final authoritative paid amount and status.

```sql
SELECT
    service_date,
    provider_id,
    COUNT(DISTINCT claim_id) AS paid_claim_count,
    SUM(paid_amount) AS total_paid_amount
FROM claims
WHERE status = 'PAID'
GROUP BY
    service_date,
    provider_id
ORDER BY
    service_date,
    provider_id;
```

---

### The 6 Critical Follow-Up Scenarios & Solutions

#### Follow-up 1: What if a claim has multiple lines?
* **Answer:** *Grain isolation.* Calculate provider daily paid totals strictly from the claim header table where `paid_amount` is adjudicated. If reporting requires line-level procedure breakdowns, pre-aggregate procedure metrics to `claim_id` before joining, or calculate line financials using line-level `service_amount`. Never sum header payments after joining unaggregated lines.

#### Follow-up 2: What if the same claim appears in two source files?
* **Answer:** Deduplicate across files using window ranking (`QUALIFY ROW_NUMBER() OVER (PARTITION BY claim_id ORDER BY updated_at DESC, ingested_at DESC) = 1`) before performing the provider aggregation.

#### Follow-up 3: What if a paid claim is subsequently reversed?
* **Answer:** Health plans issue negative adjustments or reversals. Model reversals as financial ledger entries.
  * *Option A (Additive Ledger):* Store reversal as a distinct row with a negative `paid_amount`. The `SUM(paid_amount)` naturally nets out to $0.
  * *Option B (Status Flag):* If marked with `status = 'REVERSED'`, apply conditional sign inversion:
    ```sql
    SUM(CASE WHEN status = 'REVERSED' THEN -paid_amount ELSE paid_amount END)
    ```

#### Follow-up 4: What if a claim's status changes from pending to paid?
* **Answer:** Handle via **SCD Type 1** in the current-state Silver table using Delta `MERGE INTO`, updating `status` and `paid_amount`. For auditing and history, maintain an immutable append-only Bronze log or **SCD Type 2** history table with `effective_date`, `end_date`, and `is_current`.

#### Follow-up 5: How would you calculate the rejected claim rate?
* **Answer:** Divide rejected claims by all claims that have completed adjudication, **strictly excluding pending claims**:
  ```sql
  COUNT(DISTINCT CASE WHEN status = 'REJECTED' THEN claim_id END) * 1.0 /
  NULLIF(COUNT(DISTINCT CASE WHEN status IN ('PAID', 'REJECTED') THEN claim_id END), 0)
  ```

#### Follow-up 6: How would you identify providers whose paid amounts increased by > 20% Month-over-Month (MoM)?
* **Answer:** Aggregate payments by provider and month, use the `LAG()` window function to retrieve previous month's paid amount, and filter for `(current - prev) / NULLIF(prev, 0) > 0.20`.


In [ ]:
# Executing Solutions to the 6 SQL Interview Follow-Up Questions

# Q1: Baseline Provider Daily Paid Summary
print("=== Q1: Provider Daily Paid Summary ===")
q1_sql = """
SELECT
    claim_date AS service_date,
    provider_id,
    COUNT(DISTINCT claim_id) AS paid_claim_count,
    ROUND(SUM(paid_amount), 2) AS total_paid_amount
FROM claims
WHERE status = 'Approved'
GROUP BY claim_date, provider_id
ORDER BY claim_date, provider_id
LIMIT 5;
"""
display(q(q1_sql))

# Q2: Deduplicating Across Source Files with QUALIFY
print("\n=== Q2: Deduplication Across Multiple Files ===")
q2_sql = """
SELECT
    claim_id,
    provider_id,
    status,
    paid_amount,
    updated_at
FROM claims
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY claim_id 
    ORDER BY updated_at DESC, ingested_at DESC
) = 1
LIMIT 5;
"""
display(q(q2_sql))

# Q3: Reversals & Net Financial Settlement
print("\n=== Q3: Reversal Handling & Net Ledger Impact ===")
q3_sql = """
WITH sample_ledger AS (
    SELECT 'CLM100' AS claim_id, 'PRV001' AS provider_id, 500.00 AS paid_amount, 'PAID' AS status
    UNION ALL
    SELECT 'CLM100' AS claim_id, 'PRV001' AS provider_id, 500.00 AS paid_amount, 'REVERSED' AS status
    UNION ALL
    SELECT 'CLM200' AS claim_id, 'PRV001' AS provider_id, 1200.00 AS paid_amount, 'PAID' AS status
)
SELECT
    provider_id,
    COUNT(DISTINCT claim_id) AS distinct_claims,
    ROUND(SUM(CASE WHEN status = 'REVERSED' THEN -paid_amount ELSE paid_amount END), 2) AS net_paid_amount
FROM sample_ledger
GROUP BY provider_id;
"""
display(q(q3_sql))

# Q5: Rejected Claim Rate Excluding Pending
print("\n=== Q5: Adjudicated Rejected Claim Rate ===")
q5_sql = """
SELECT
    ROUND(
        COUNT(CASE WHEN status = 'Denied' THEN 1 END) * 100.0 /
        NULLIF(COUNT(CASE WHEN status IN ('Approved', 'Denied') THEN 1 END), 0),
        2
    ) AS rejected_rate_pct
FROM claims;
"""
display(q(q5_sql))

# Q6: Providers with > 20% Month-over-Month (MoM) Growth
print("\n=== Q6: Providers with > 20% Month-over-Month Growth ===")
q6_sql = """
WITH monthly_provider_paid AS (
    SELECT
        provider_id,
        DATE_TRUNC('month', claim_date) AS service_month,
        ROUND(SUM(paid_amount), 2) AS monthly_paid
    FROM claims
    WHERE status = 'Approved'
    GROUP BY provider_id, DATE_TRUNC('month', claim_date)
),
mom_lagged AS (
    SELECT
        provider_id,
        service_month,
        monthly_paid,
        LAG(monthly_paid) OVER (
            PARTITION BY provider_id 
            ORDER BY service_month
        ) AS prev_month_paid
    FROM monthly_provider_paid
)
SELECT
    provider_id,
    service_month,
    monthly_paid,
    prev_month_paid,
    ROUND(((monthly_paid - prev_month_paid) / NULLIF(prev_month_paid, 0)) * 100, 2) AS mom_growth_pct
FROM mom_lagged
WHERE prev_month_paid > 0
  AND ((monthly_paid - prev_month_paid) / NULLIF(prev_month_paid, 0)) > 0.20
ORDER BY mom_growth_pct DESC
LIMIT 5;
"""
display(q(q6_sql))


---
## Section 7: PySpark Interview Exercise & Execution Internals

### The PySpark Code
```python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

window_spec = (
    Window
    .partitionBy("claim_id")
    .orderBy(
        F.col("updated_at").desc(),
        F.col("ingestion_timestamp").desc(),
        F.col("_file_row_offset").desc() # Production deterministic tie-breaker
    )
)

deduplicated_df = (
    claims_df
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)
```

---

### The 3 Core Execution Internals Interviewers Test

#### 1. `Window.partitionBy("claim_id")` vs Physical Repartitioning
* **Key Concept:** `Window.partitionBy("claim_id")` does **NOT** physically repartition the Spark DataFrame into new cluster partition files.
* **Under the Hood:** Spark introduces an `Exchange hashpartitioning(claim_id, 200)` shuffle stage, routing all records with the same `claim_id` to the same executor buffer, followed by `SortExec` and `SortWindowExec`.
* **Contrast:** Physical `df.repartition("claim_id")` permanently redistributes DataFrame partitions across the cluster, which is expensive and unnecessary here.

#### 2. `row_number()` vs `rank()` vs `dense_rank()`
* `row_number()` assigns a sequential integer `[1, 2, 3...]` strictly breaking ties based on sort order. It guarantees **exactly one row** is retained.
* `rank()` / `dense_rank()` assign equal ranks to tied rows `[1, 1, 3...]`. If two records share the same timestamps, `filter(row_num == 1)` would retain **both copies**, defeating the entire deduplication step!

#### 3. Deterministic Secondary Tie-Breakers
* In high-throughput distributed systems, multiple claim versions may carry identical `updated_at` and `ingestion_timestamp`.
* **Production Requirement:** Always append a deterministic tie-breaker (e.g. `_file_row_offset`, source file path, or record UUID) to guarantee 100% deterministic, reproducible output across pipeline retries.

#### Claim Header vs. Claim Line Deduplication Grain:
* **Header:** Deduplicate on `claim_id`.
* **Line:** Deduplicate on composite key `(claim_id, line_id)`! Deduplicating lines on `claim_id` alone would destroy legitimate service lines.


In [ ]:
# Executing Window Deduplication in PySpark & Plan Inspection

# Create a test DataFrame with deliberate duplicate claim versions
sample_claims_data = [
    ("CLM9001", "PRV10", "2026-10-01", 1200.00, "2026-10-01 10:00:00", "2026-10-01 10:05:00", 1),
    ("CLM9001", "PRV10", "2026-10-01", 1200.00, "2026-10-01 11:30:00", "2026-10-01 11:35:00", 2), # Latest
    ("CLM9002", "PRV20", "2026-10-01", 450.00,  "2026-10-01 08:00:00", "2026-10-01 08:05:00", 1),
    ("CLM9002", "PRV20", "2026-10-01", 450.00,  "2026-10-01 08:00:00", "2026-10-01 08:05:00", 2), # Identical timestamps, offset tie-breaker
]

schema = ["claim_id", "provider_id", "service_date", "paid_amount", "updated_at", "ingested_at", "_offset"]
raw_spark_df = spark.createDataFrame(sample_claims_data, schema)

# Deduplication Window Spec
dedup_window = (
    Window.partitionBy("claim_id")
    .orderBy(
        F.col("updated_at").desc(),
        F.col("ingested_at").desc(),
        F.col("_offset").desc()
    )
)

deduped_spark_df = (
    raw_spark_df
    .withColumn("row_num", F.row_number().over(dedup_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

print("--- Raw Incoming Records (4 rows) ---")
display(raw_spark_df.toPandas())

print("\n--- Deduplicated Output (2 rows) ---")
display(deduped_spark_df.toPandas())

print("\n--- Spark Physical Execution Plan (SortWindowExec) ---")
deduped_spark_df.explain(True)


---
## Section 8: Incremental Loads, Delta MERGE, and Idempotency

### The Interview Question
> *"Imagine an orchestration DAG fails after processing 70% of today's 3 TB claims. When you rerun the job, how do you guarantee you do not duplicate payments or double-count claims?"*

---

### The 5 Architectural Pillars of Idempotency

1. **Auto Loader Checkpointing:**
   * Databricks Auto Loader maintains an internal streaming transaction log and RocksDB state store.
   * If a task fails mid-batch, the checkpoint is **never committed**. The subsequent run safely picks up unprocessed S3 files without missing or duplicating files.

2. **Stable Business Key Matching:**
   * Upserts match strictly on the business key (`target.claim_id = source.claim_id`).

3. **Delta Lake `MERGE INTO` (ACID Atomicity):**
   ```sql
   MERGE INTO silver.claims AS target
   USING staged_claims AS source
   ON target.claim_id = source.claim_id
   WHEN MATCHED AND source.updated_at >= target.updated_at THEN
     UPDATE SET *
   WHEN NOT MATCHED THEN
     INSERT *;
   ```
   * Delta transactions are all-or-nothing (Atomicity). If the merge fails at 70%, uncommitted parquet files are ignored by subsequent transactions and cleaned up by `VACUUM`.

4. **The Source Deduplication Prerequisite:**
   > [!WARNING]
   > Delta Lake `MERGE INTO` will fail with an error:
   > `Cannot perform Merge as multiple source rows matched the same target row`
   > If the source batch contains duplicate `claim_id`s. **Source deduplication is mandatory before calling MERGE!**

5. **Databricks Workflows "Repair and Rerun":**
   * If Task 4 fails, you do not rerun the entire 8-hour pipeline. Databricks Workflows allows **repairing and rerunning only the failed task**, preserving all completed upstream Bronze and Silver tasks.


In [ ]:
# Demonstrating Idempotent Delta Upsert / Merge Logic

# 1. Initial State of Target Silver Table
target_claims = pd.DataFrame([
    {"claim_id": "CLM101", "status": "Submitted", "paid_amount": 0.00, "updated_at": "2026-10-01 10:00:00"},
    {"claim_id": "CLM102", "status": "Approved", "paid_amount": 450.00, "updated_at": "2026-10-01 10:00:00"},
])

# 2. Incoming Batch (Contains an update to CLM101, a replay of CLM102, and a new CLM103)
incoming_batch = pd.DataFrame([
    {"claim_id": "CLM101", "status": "Approved", "paid_amount": 850.00, "updated_at": "2026-10-01 15:00:00"}, # Valid update
    {"claim_id": "CLM102", "status": "Approved", "paid_amount": 450.00, "updated_at": "2026-10-01 10:00:00"}, # Duplicate replay
    {"claim_id": "CLM103", "status": "Approved", "paid_amount": 320.00, "updated_at": "2026-10-01 12:00:00"}, # New claim
])

def idempotent_merge(target_df, source_df):
    """Simulates Delta MERGE INTO target USING source ON claim_id"""
    merged = pd.concat([target_df, source_df], ignore_index=True)
    # Deduplicate keeping latest updated_at
    merged["updated_at"] = pd.to_datetime(merged["updated_at"])
    deduped = merged.sort_values(by=["updated_at"], ascending=False).drop_duplicates(subset=["claim_id"], keep="first")
    return deduped.sort_values(by=["claim_id"]).reset_index(drop=True)

# Run Merge Once
run_1 = idempotent_merge(target_claims, incoming_batch)
# Run Merge a Second Time (Simulating Job Retry)
run_2 = idempotent_merge(run_1, incoming_batch)

print("--- Target State After Run 1 ---")
display(run_1)

print("\n--- Target State After Run 2 (Retry Verification) ---")
display(run_2)

# Verify Idempotency: Run 1 and Run 2 must be identical in rows and payment sums
is_identical = run_1.equals(run_2)
total_paid_1 = run_1["paid_amount"].sum()
total_paid_2 = run_2["paid_amount"].sum()

print(f"\nIdempotency Check: {is_identical} (Run 1 Paid: ${total_paid_1:,.2f} == Run 2 Paid: ${total_paid_2:,.2f})")
assert is_identical and total_paid_1 == total_paid_2, "FATAL: Pipeline retry modified financial state!"
print("SUCCESS: Pipeline retry is 100% idempotent.")


---
## Section 9: Performance Optimization (Reducing 8 Hours to Under 30 Minutes on 3 TB)

### The Interview Question
> *"Your 3 TB claims pipeline currently takes 8 hours to run. How would you systematically reduce the runtime to under 30 minutes?"*

---

### The 6-Step Optimization Protocol (Never Guess Compute First!)

```mermaid
graph TD
    S1[1. Measure First: Spark UI, Shuffle Spill, GC, Skew] --> S2[2. Reduce Scan: Column Pruning & Partition Pruning]
    S2 --> S3[3. Optimize Joins: Broadcast Small Dimensions, Mitigate Skew]
    S3 --> S4[4. Storage Layout: Liquid Clustering / Z-Order & File Compaction]
    S4 --> S5[5. Eliminate Redundant Work: Incremental Auto Loader & Delta CDF]
    S5 --> S6[6. Benchmark & Verify: Measure DBU cost & runtime against baseline]
```

#### Step 1: Measure First (Spark UI Analysis)
* Open the **Spark UI Stage Details**:
  * Check **Shuffle Spill (Memory vs Disk)**: If tasks spill gigabytes to disk, executors are thrashing. Increase partition count (`spark.sql.shuffle.partitions`) or executor memory.
  * Check **Task Duration Distribution**: If 199 tasks finish in 20 seconds and 1 task takes 2.5 hours, you have severe **Data Skew** (e.g. mega-provider or null keys).
  * Check **GC Time**: If GC > 15% of task time, tune JVM garbage collection or reduce object creation.

#### Step 2: Reduce Unnecessary Data Scanned
* **Column Pruning:** Drop unneeded raw JSON payloads and audit metadata before shuffling.
* **Predicate Pushdown:** Partition/cluster by `service_date` so queries scan only the current processing window.

#### Step 3: Optimize Joins
* **Broadcast Join:** `providers` (50,000 rows) and `members` dimension tables must be **broadcast** (`F.broadcast(providers)`). This eliminates a 3 TB shuffle exchange entirely!
* **Skew Join Handling:** Enable Adaptive Query Execution (`spark.sql.adaptive.skewJoin.enabled = true`) to automatically split skewed partitions.

#### Step 4: Storage Optimization (Liquid Clustering & Compaction)
* Run `OPTIMIZE silver.claims CLUSTER BY (service_date, provider_id);`
* Replaces static hierarchical directory partitioning with Databricks **Liquid Clustering**, preventing small-file proliferation and speeding up join lookups.

#### Step 5: Eliminate Redundant Scans
* Never reprocess the full 3 TB historical dataset daily. Use **Delta Change Data Feed (CDF)** and **Auto Loader** to process only the daily net changes.


In [ ]:
# Demonstrating Join Optimization: SortMergeJoin vs BroadcastHashJoin

claims_spark = spark.table("claims")
providers_spark = spark.table("providers")

# 1. Standard Join
standard_join_df = claims_spark.join(providers_spark, on="provider_id", how="inner")

# 2. Broadcast Join (Eliminates Shuffle Exchange)
broadcast_join_df = claims_spark.join(F.broadcast(providers_spark), on="provider_id", how="inner")

print("=== Execution Plan: Standard Join ===")
standard_plan = plan_string(standard_join_df)
print("\n".join([line for line in standard_plan.split("\n") if "Join" in line or "Exchange" in line or "Broadcast" in line][:4]))

print("\n=== Execution Plan: Optimized Broadcast Join (Zero Shuffle) ===")
broadcast_plan = plan_string(broadcast_join_df)
print("\n".join([line for line in broadcast_plan.split("\n") if "Join" in line or "Broadcast" in line][:4]))

print(f"\nHas Broadcast Hash Join: {has_plan_operator(broadcast_join_df, 'BroadcastHashJoin') or has_plan_operator(broadcast_join_df, 'Broadcast')}")
print("Summary: Broadcasting the provider dimension eliminated the 3 TB shuffle exchange on the cluster!")


---
## Section 10: Databricks Workflows Production Job Definition & Senior DE Interview Playbook

### Why Databricks Workflows over Standalone Airflow for Lakehouse Workloads?
1. **Serverless & Zero Infrastructure:** Eliminates Airflow Celery worker management, database metadata maintenance, and worker OOM crashes.
2. **Native Unity Catalog & Lineage:** Automatic end-to-end data lineage tracking down to column-level granularity.
3. **Repair and Rerun:** Ability to restart a multi-task pipeline from the exact point of failure without rerunning successful upstream tasks.
4. **Task Parameter Passing (`taskValues`):** Native key-value parameter passing between DAG tasks without external Redis/XCom bottlenecks.

---

### Production Databricks Workflows Multi-Task Job Specification (JSON)

```json
{
  "name": "claims_processing_medallion_pipeline",
  "email_notifications": {
    "on_failure": ["data-engineering-oncall@healthcare.com"]
  },
  "webhook_notifications": {
    "on_failure": [{"id": "pagerduty-claims-critical"}]
  },
  "tasks": [
    {
      "task_key": "ingest_s3_to_bronze",
      "notebook_task": {
        "notebook_path": "/Workspace/claims/01_ingest_s3_to_bronze",
        "base_parameters": {"batch_date": "{{workflow.start_time}}"}
      },
      "max_retries": 2,
      "min_retry_interval_millis": 30000
    },
    {
      "task_key": "quarantine_and_silver_merge",
      "depends_on": [{"task_key": "ingest_s3_to_bronze"}],
      "notebook_task": {
        "notebook_path": "/Workspace/claims/02_quarantine_and_silver_merge"
      },
      "max_retries": 1
    },
    {
      "task_key": "adjudicate_reversals_referential",
      "depends_on": [{"task_key": "quarantine_and_silver_merge"}],
      "notebook_task": {
        "notebook_path": "/Workspace/claims/03_adjudicate_reversals"
      }
    },
    {
      "task_key": "build_gold_daily_marts",
      "depends_on": [{"task_key": "adjudicate_reversals_referential"}],
      "notebook_task": {
        "notebook_path": "/Workspace/claims/04_build_gold_marts"
      }
    },
    {
      "task_key": "dq_reconciliation_gatekeeper",
      "depends_on": [{"task_key": "build_gold_daily_marts"}],
      "notebook_task": {
        "notebook_path": "/Workspace/claims/05_dq_reconciliation_gatekeeper"
      }
    },
    {
      "task_key": "alert_on_reconciliation_failure",
      "depends_on": [{"task_key": "dq_reconciliation_gatekeeper"}],
      "condition_task": {
        "op": "EQUAL_TO",
        "left": "{{tasks.dq_reconciliation_gatekeeper.values.reconciliation_status}}",
        "right": "FAILED"
      },
      "notebook_task": {
        "notebook_path": "/Workspace/claims/06_trigger_pagerduty_incident"
      }
    }
  ]
}
```

---

### Senior Data Engineer Interview Cheatsheet (The 2-Minute Pitch)

> **"If an interviewer asks: 'Walk me through your claims data pipeline from end to end':"**
>
> 1. **Ingestion (Bronze):** *"We ingest approximately 3 TB of daily claim JSON/EDI files from S3 landing into Delta Bronze using Databricks Auto Loader, appending ingestion metadata and tracking schema evolution without downtime."*
> 2. **Validation & Deduplication (Silver):** *"In Silver, we enforce schema and contract rules, splitting invalid records into a quarantine table. We deduplicate incoming claims using a PySpark window function on `claim_id` ordering by `updated_at DESC` with a deterministic record offset tie-breaker. Cleansed records are transactionally upserted into Silver via Delta `MERGE INTO`."*
> 3. **Grain & Aggregations (Gold):** *"We maintain strict grain isolation: claim headers hold adjudicated payment totals while service lines hold procedure details. We build provider daily rollup marts without fan-out bugs."*
> 4. **Financial Reconciliation:** *"Before publishing to Gold, our DQ Gatekeeper task runs a three-way reconciliation ledger comparing source ingested dollars vs target ledger dollars. Any unexplained variance halts publication and triggers a PagerDuty alert."*
> 5. **Orchestration & Performance:** *"We orchestrate via Databricks Workflows with automated retries and Repair-and-Rerun capabilities. We compressed the runtime from 8 hours to under 30 minutes by broadcasting dimension joins, tuning shuffle partitions, and implementing Liquid Clustering on `(service_date, provider_id)`."*


In [ ]:
# Simulating Databricks Workflows TaskValues Passing & DQ Gatekeeper Alerting

class DatabricksTaskValuesSimulator:
    """Simulates dbutils.jobs.taskValues across pipeline tasks"""
    def __init__(self):
        self._store = {}
    
    def set(self, key, value):
        self._store[key] = value
        print(f"[TaskValue SET] {key} = {value}")
        
    def get(self, key, default=None):
        return self._store.get(key, default)

task_values = DatabricksTaskValuesSimulator()

print("=== Databricks Workflows Multi-Task Simulation ===")

# Task 1: Ingestion
print("\n--- Task 1: ingest_s3_to_bronze ---")
task_values.set("source_batch_id", "BATCH_2026_10_11_001")
task_values.set("source_claims_count", 1000000)
task_values.set("source_gross_paid", 12500000.00)

# Task 2: Quarantine & Silver Merge
print("\n--- Task 2: quarantine_and_silver_merge ---")
task_values.set("quarantined_records_count", 2800)
task_values.set("silver_merged_claims_count", 997200)

# Task 4: Gold Marts & Reconciliation
print("\n--- Task 4: build_gold_daily_marts ---")
gold_total_paid = 12490000.00 # Target paid
task_values.set("gold_total_paid", gold_total_paid)

# Task 5: DQ Gatekeeper Task
print("\n--- Task 5: dq_reconciliation_gatekeeper ---")
src_paid = task_values.get("source_gross_paid")
gold_paid = task_values.get("gold_total_paid")
expected_adjustments = 10000.00 # Quarantined ($2,400) + Deduplicated ($7,600)

unexplained_variance = abs((src_paid - expected_adjustments) - gold_paid)

if unexplained_variance == 0.00:
    task_values.set("reconciliation_status", "PASSED")
    print("[GATEKEEPER] Financial Reconciliation PASSED! Gold Marts approved for BI publishing.")
else:
    task_values.set("reconciliation_status", "FAILED")
    print(f"[GATEKEEPER] ALERT! Financial discrepancy of ${unexplained_variance:,.2f} detected! Halting release.")

# Task 6: Conditional Alert Trigger
print("\n--- Task 6: alert_incident_webhook (Conditional Task) ---")
status = task_values.get("reconciliation_status")
if status == "FAILED":
    print("[ALERT] Triggering PagerDuty incident to data-engineering-oncall@healthcare.com")
else:
    print(f"[TASK SKIPPED] Condition not met (Status={status}). Pipeline execution complete.")
